# Writing and reading columnar data
This is a Python notebook in which you will practice the concepts learned during the lectures.

## Startup ROOT
Import the ROOT module: this will activate the integration layer with the notebook automatically

In [1]:
import ROOT
import numpy as np
print(ROOT.gROOT.GetVersion())

6.38.00


## Writing and Reading NTuple
Create a **TFile** containing an `TNtuple` `(x,y,z,t)` filled with random numbers distributed according to a uniform, gaus, exponential and a Landau distribution.
Close the file: you will reopen it later.

In [2]:
rndm = ROOT.TRandom3(1)

filename = "ntuple_example.root"

# Create the File
file = ROOT.TFile.Open(filename, "RECREATE")

# Here create the Ntuple. The ntuple must be stored as "n" with the arrays "x:y:z:t"
Ntuple = ROOT.TNtuple("n", "ntuple example", "x:y:z:t")

# Here create the arrays x,y,z and t and assign random values
n=10000
# x is a uniform between -10 and 10
x=np.random.uniform(-10,10,n)
# y is a normal with mean=0 and sigma=5
y=np.random.normal(0,5,n)
# z is an exponential with scale=10
z=np.random.exponential(10,n)

t=np.array(list(map(lambda i: rndm.Landau(0,2),range(n))),'f')

# Fill the ntuple
for i in range(n):
    # fill the ntuple
    Ntuple.Fill(x[i], y[i], z[i], t[i])

# Here write the ntuple on the file and close the file
Ntuple.Write()
file.Close()

Now, you can invoke the *ls* command from within the notebook to list the files in this directory. Check that the file is there. You can invoke the *rootls* command to see what's inside the file and using the `-t` option to print-out a detailed output  

In [3]:
! ls .
! echo Now listing the content of the file
! rootls -t ntuple_example.root 

Hgg.txt		      param_fitting_part1.html	 rdataframe.ipynb
columnar_data.ipynb   param_fitting_part1.ipynb	 ttree_example.root
debugging.ipynb	      param_fitting_part2.html	 vector_collection.root
extra_higgs2gg.ipynb  param_fitting_part2.ipynb	 writing_files.html
histos.root	      plotting_insights.html	 writing_files.ipynb
ntuple_example.root   plotting_insights.ipynb
Now listing the content of the file
TNtuple  Jan 15 16:47 2026 n;1 "ntuple example" 
  x  "x"  40130
  y  "y"  40130
  z  "z"  40130
  t  "t"  40130
  Cluster INCLUSIVE ranges:
   - # 0: [0, 9999]
  The total number of clusters is 1


Access the created ntuple and draw all the elements. Remember that you need to create a TCanvas before and draw it too in order to inline the plots in the notebooks.

In [4]:
# Activam la visualització interactiva
%jsroot on

# Accedim a l'arxiu i representem el contingut de l'ntuple
f = ROOT.TFile(filename)
c = ROOT.TCanvas()
c.Divide(2,2)
c.cd(1)

Ntuple = f.Get("n")

Ntuple.Draw("x>>h_gaus(100,-20,20)","","COLZ")
c.cd(2)
Ntuple.Draw("y>>h_exp(100,-20,20)","","COLZ")
c.cd(3)
Ntuple.Draw("z>>h_uniform(100,0,50)","","COLZ")

c.Draw()

Read the ntuple event by event and fill an histogram with the `x*y` of each event, only if `t` is positive

In [5]:
h=ROOT.TH1F("h","h",100,-100,100)
# Loop over all the events
for evt in Ntuple:
    if evt.t > 0:
        # Fill the histo with x*y       
        h.Fill(evt.x*evt.y) 
c.cd(4)
h.Draw()
c.Draw()

## Writing and reading TTree
### TTree with plain types
We are using the same values than in the previous `TNtuple` section to create a `TTree`. 

**IMPORTANT NOTE**: A TTree uses variables to fill it, which assigns to the corresponding branch. The assignment is done using pointers in C++, which facilitates the serialization of the object which is going to be stored. In python, a plain type (int, floats, ...) needs to mimic the address assignation by using a list, array or any python container. This can be accomplished by `array.array` or event better by `numpy.array`.

Close the file: you will reopen it later.

In [6]:
filename_tree = "ttree_example.root"

# Open the file
ftree=ROOT.TFile(filename_tree,"RECREATE")

# Here create the TTree
tree=ROOT.TTree("tree","Tree with plain types")

# Declare the variables to be assigned to the TTree
# Remember to use arrays and initialize at least 1 element
az = np.array([0],'f') # z variable
at = np.array([0],'f') # t variable
N = 10000

# Assign the variables to the branches
# we are using same names for the branches: x,y,z,t 
tree.Branch("t",at,"t/F")

# Fill the TTree
for i in range(N):
    # Assign the values to the corresponding variables,
    # Use the vectors already filled: x[i], y[i], z[i] and t[i]
    az[0] = z[i]
    at[0] = t[i]

    # Remember to fill the tree before ending
    tree.Fill()

# Here write the ntuple on the file and close the file
tree.Write()
ftree.Close()


Access the created tree and draw all the elements. You can re-use the code from the ntuple.

In [7]:
f2=ROOT.TFile(filename_tree)
c2=ROOT.TCanvas()
# Create all the 4 pads and draw the same histograms than in the TNtuple case
c2.Divide(2,2)
c2.cd(1)
tree = f2.Get("tree")
Ntuple.Draw("x>>h_gaus(100,-20,20)","","COLZ")
c2.cd(2)
Ntuple.Draw("y>>h_exp(100,-20,20)","","COLZ")
c2.cd(3)
Ntuple.Draw("z>>h_uniform(100,0,50)","","COLZ")
c2.cd(4)
tree.Draw("t>>h_landau(100,-10,30)","","COLZ")
c2.Draw()

### TTree with generic objects
So far, we did not use the full potential of the `TTree`. Any object could be stored in a TTree, not only plain types. Let's store a similar information than before, but in a more compact way using STL vectors

In [ ]:
filename_tree = "ttree_example.root"

# Update the file with a new tree but without destroy previous data
ftree=ROOT.TFile.Open(filename_tree,"UPDATE")

# Here create the new TTree
tree2=ROOT.TTree("tree_vector","Tree with complex objects")

# Declare the variables to be assigned to the TTree: a 4-dimensional vector mimicking tracks
pt_vec=ROOT.std.vector(float)()
phi_vec=ROOT.std.vector(float)()
eta_vec=ROOT.std.vector(float)()
mass_vec=ROOT.std.vector(float)()

# Declare a helper list
vec_list= [ pt_vec, phi_vec, eta_vec, mass_vec ]

# Assign those variables to the TTree branches, remember use TTree.Branch method
for i, var in enumerate( ["pt","phi","eta","mass"] ):
    tree2.Branch( var, vec_list[i] )
                
# Create N-random pions
mass=0.13957

N = 10000

# How many pions per event (20 in average)
npart = np.random.poisson(20,N)


# Fill the TTree
for i in range(N):
    # clear the vectors and prepare them (use reserve) for the next iteration
    # IMPORTANT!! Clear the vectors, otherwise you are storing all the accumulated info

    n_pions = int(npart[i])
    for vec in vec_list:
        vec.clear()
        vec.reserve(n_pions)
    
    # Create the pions of the current event
    pt=np.random.exponential(10,n_pions)
    eta=np.random.uniform(-3,3,n_pions)
    phi=np.random.uniform(0,2.0*np.pi,n_pions)

    # Store each particle in the vector
    for j in range(n_pions):
        pt_vec.push_back(pt[j])
        eta_vec.push_back(eta[j])
        phi_vec.push_back(phi[j])
        mass_vec.push_back(mass)

    # fill the tree
    tree2.Fill()

# Here write the ntuple on the file and close the file
tree2.Write()
ftree.Close()

As extra you can try to plot again all the values stored and compare them with the previous plain type example

In [9]:
# Let's try to plot again all the values stored in the tree_vector TTree and compare with previous results
filename_tree = "ttree_example.root"
# Open the file
ftree2=ROOT.TFile.Open(filename_tree,"READ")

# Create a canvas
c3=ROOT.TCanvas()
# Create all the 4 pads and draw the same histograms than in the TNtuple case
c3.Divide(2,2)
c3.cd(1)

tree2 = ftree2.Get("tree_vector")
tree2.Draw("pt>>h_pt(100,0,60)","","HIST")
c3.cd(2)
tree2.Draw("phi>>h_phi(100,0,7)","","HIST")
c3.cd(3)
tree2.Draw("eta>>h_eta(100,-4,4)","","HIST")
c3.cd(4)
tree2.Draw("@pt.size()>>h_npart(40,0,40)","","HIST")
c3.Draw()

### Reading TTree with generic objects
The reading exercise is going to be prepared using the example from the slide `"Filling a Tree with Objects"`. Execute the next cell to create the input file you need

In [10]:
fname_objects="vector_collection.root"
f=ROOT.TFile(fname_objects,"RECREATE")
t=ROOT.TTree("tree_tracks","Tree with pseudo-particles")
particles= ROOT.std.vector(ROOT.std.vector(float))()

t.Branch("tracks", particles)

# pi+/pi- mass
mass = 0.13957  
for i in range(5000):
    nPart = rndm.Poisson(20)
    particles.clear(); particles.reserve(nPart)
    for j in range(nPart):
        cpar=ROOT.std.vector(float)();
        pt = rndm.Exp(10); eta = rndm.Uniform(-3,3); phi = rndm.Uniform(0, 2*ROOT.TMath.Pi())
        cpar.push_back(pt); cpar.push_back(eta); cpar.push_back(phi); cpar.push_back(mass);
        particles.push_back(cpar)
    t.Fill()
t.Write()
f.Close()

Retrieve the vector of particles, i.e., a 4-vector with (pt.eta,phi,mass), and plot the **momentum** (not pt!!), the **eta**, the **total energy available at each event** and the **number of particles per event**. 

Trick: you can obtain in [wikipedia](https://en.wikipedia.org/wiki/Pseudorapidity) the definition of eta (pseudorapidity) and how to relate the transverse momentum, eta and phi with the cartesian momentum.

In [11]:
# Open the file in read-only mode
f=ROOT.TFile.Open(fname_objects)
# Obtain the tree where tracks are, "tree_tracks"
t = f.Get("tree_tracks")

# Create the needed histograms
hmom=ROOT.TH1F("hmom","Momentum;p [GeV/c];Entries",100,0,150)
heta=ROOT.TH1F("eta","#eta",100,-3.3,3.3)
henergy=ROOT.TH1F("total energy","Energy per event;#SigmaE [GEV/c^{2}];Events",100,0,2500)
hn=ROOT.TH1I("total particles","Pions per event;N_{pions};Events}",50,-0.5,49.5)

# Some useful functions
# momentum (use a lambda function to obtain the momentum of each particle)
momentum = lambda p: ROOT.TMath.Sqrt( p[0]*p[0] * (1 + ROOT.TMath.SinH(p[1])*ROOT.TMath.SinH(p[1])) )

# energy (use a lambda function to obtain the energy of each particle)
energy = lambda p: ROOT.TMath.Sqrt(momentum(p)*momentum(p) + p[3]*p[3])

# Remember: each element of the vector tracks contain a 4-vector
# (pt,eta,phi,mass)

# Loop over particles
for evt in t:
    # Obtain the total number of particles in the event
    ntot=evt.tracks.size()
    
    # Loop over the particles to fill 
    etot=0.0
    for j in range(ntot):
        # Calculate the momentum and fill the momentum histo
        hmom.Fill(momentum(evt.tracks[j]))
        # Fill the eta histogram
        heta.Fill(evt.tracks[j][1])
        
        # Sum-up the contribution of this particle in the energy
        # (Note: here is using the lambda function implemented before)
        etot += energy(evt.tracks[j])
    # Fill the "per event" histograms
    hn.Fill(ntot)
    henergy.Fill(etot)

Finally plot all the histograms: momentum, eta, energy and number of particles

In [12]:
f4 = ROOT.TFile(fname_objects)
c4 = ROOT.TCanvas()
# As usual ...
c4.Divide(2,2)
c4.cd(1)
hmom.Draw()
c4.cd(2)
heta.Draw()
c4.cd(3)
henergy.Draw()
c4.cd(4)
hn.Draw()
c4.Draw()